# E-COMMERCE ORDERING SYSTEM
You will build a simplified E-Commerce Order Management System. The system handles two types of products — Clothing and Books — and allows customers to add items to a shopping cart and place a purchase directly from it.
## CLASS STRUCTURE
- `Product` (Parent Class)
- `Clothing` (Child Class)
- `Book`     (Child Class)
- `ShoppingCart` (Uses Product objects)

# PART A — Product Class (Base Class)
The Product class is the parent class for all products in the system.
Attributes
- product_id (str) — Unique ID, e.g. "P001"
- name (str) — Product name
- price (float) — Price in Rs.
- stock_quantity (int) — Available stock
- category (str) — Product category

## Methods Method Description
`__init__(...)`: Initialize all 5 attributes listed above. 
`__str__()`:  Return: `"[Name] (Rs.[Price]) - Stock: [Stock]"`
              Example: `"Cotton T-Shirt (Rs.500) - Stock: 50"`
`check_availability(qty)`: Return __True__ if `stock_quantity >= qty`, else __False__.
`update_stock(qty)`: Reduce stock by qty. Validate: qty > 0 AND stock is sufficient. On success → print: __Stock updated__. Remaining: [stock_quantity] On failure → print appropriate error message. 
`calculate_total(qty)`: Return price × qty. (Will be overridden by child classes.) 
`display_info()`: Print full product details. 
See format below
## display_info() output format:
Product ID : P001\
Name : Cotton T-Shirt\
Category : Clothing\
Price : Rs.500\
Stock : 50 units

In [22]:
class Product:
    def __init__(self, prodid, name, price, stock_quantity, category):
        self.prodid=prodid
        self.name=name
        self.price=price
        self.stock_quantity=stock_quantity
        self.category=category
    def get_price(self):
        return self.price
    def __str__(self):
        return f"""
                {self.name} (Rs. {self.price}) - Stock: {self.stock_quantity}"""
    def check_availability(self, qty):
        if self.stock_quantity >= qty:
            return True
        else:
            return False
    def stock_update(self, qty):
        if qty>0 and self.stock_quantity >= qty:
            self.stock_quantity -= qty
            return f"Stock updated. Remaining: {self.stock_quantity}"
        else:
            return f"Insufficient Stock."
    def calculate_total(self, qty):
        return self.price * qty
    def display_info(self):
        return f"""
                Product ID : {self.prodid}
                Name       : {self.name}
                Category   : {self.category}
                Price      : {self.price}
                Stock      : {self.stock_quantity} units"""           

# PART B — Clothing Class (Inherits from Product)
Clothing inherits from Product and adds size and material information.
## Additional Attributes
- `size (str)` — e.g. "S", "M", "L", "XL", "XXL"
- `material (str)` — e.g. "Cotton", "Denim"
## Methods Method Description
- `__init__(...)` Call parent `__init__` with category set to "Clothing". Then set size and material.
- `calculate_total(qty)` Override: Add Rs.100 stitching charge per item for XL and XXL sizes. Inherit base cost from parent first.
- `display_info()` use parent’s display_info().then print Size and Material below it.
### display_info() output format:
Product ID : C001\
Name : Denim Jeans\
Category : Clothing\
Price : Rs.1500\
Stock : 30 units\
Size : XL\
Material : Denim

In [23]:
class Clothing(Product):
    def __init__(self, prodid, name, price, stock_quantity, size, material):
        super().__init__(prodid, name, price, stock_quantity, "Clothing")
        self.size=size
        self.material=material
    def calculate_total(self, qty):
        parent_price=super().get_price()
        if self.size in ("XL", "XXL"):
            parent_price += 100
        return parent_price * qty
    def display_info(self):
        return f"""
                {super().display_info()}
                Size       : {self.size}
                Material   : {self.material}""" 

# PART C — Book Class (Inherits from Product)
Book inherits from Product and adds author, ISBN, and page count.
## Additional Attributes
- author (str) — Author name
- isbn (str) — ISBN number, e.g. "978-1234567890"
- pages (int) — Number of pages
## Methods
Method Description
- `__init__(...)` Call parent __init__ with category set to "Books". Then set author, isbn, and pages.
- `calculate_total(qty)` Override: Apply a 10% discount for bulk orders of 5 or more books. Inherit base cost from parent first.
- `display_info()` use parent’s display_info(), then print Author, ISBN, and Pages below it.
### display_info() output format:
Product ID : B001\
Name : Python Programming\
Category : Books\
Price : Rs.450\
Stock : 100 units\
Author : John Doe\
ISBN : 978-1234567890\
Pages : 350

In [24]:
class Book(Product):
    def __init__(self, prodid, name, price, stock_quantity, author, isbn, pages):
        super().__init__(prodid, name, price, stock_quantity, "Books")
        self.author=author
        self.isbn=isbn
        self.pages=pages
    def calculate_total(self, qty):
        parent_price=super().get_price()
        if qty >= 5:
            parent_price -= (0.1 * parent_price)
        return parent_price * qty
    def display_info(self):
        return f"""
                {super().display_info()}
                Author     : {self.author}
                ISBN       : {self.isbn}
                Pages      : {self.pages}"""            

# PART D — ShoppingCart Class
The `ShoppingCart` class lets customers add products and place a purchase.
## Private Attribute
- items(dict) — Keys are Product objects, values are quantities.
## Methods Method Description
- `__init__(self)`: Initialize itemsas an empty dictionary.
- `add_item(product, qty)`: Validate: qty > 0 and product.check_availability(qty). If product already in cart → increase quantity. If new product → add it to cart. Print: Added [name] x [qty] to cart remove_item(product) If product is in cart → remove and print confirmation. If not found → print: [name] not found in cart
- `get_total()`: Loop through all items and call product.calculate_total(qty) for each. Return the sum total. (each product type calculates differently.) checkout() Place the purchase directly from the cart. Validate: cart is not empty. Print all items with quantity and cost. Ask “Proceed to Checkout?” If yes → For each item: call product.update_stock(qty). Print: Purchase successful! Total paid: Rs.[total] Then delete the dictionary.  
### checkout() output format:
CART SUMMARY\
Cotton T-Shirt x 3 = Rs.1500\
Python Programming x 6 = Rs.2430.0\
----------------------------------------\
    Total: Rs.3930.0\
----------------------------------------\
Proceed to checkout? (y/n): y\
Stock updated. Remaining: 47\
Stock updated. Remaining: 94\
Order placed successfully! Amount paid: Rs.3930.0\
Cart cleared

In [25]:
class ShoppingCart:
    def __init__(self):
        self.__items={}
    def add_item(self, product, qty):
        if qty <=0 :
            print("Quantity must be greater than 0.")
            return
        if product.check_availability(qty) == False:
            print(f"Sorry, insufficient stock for {product.name}")
            return
        if product in self.__items:
            self.__items[product]+=qty
        else:
            self.__items[product]=qty
        return f"Added {product.name} x {qty} to cart."
    def remove_item(self, product):
        if product not in self.__items:
            print(f"{product.name} not found in cart.")
            return
        else:
            self.__items.pop(product)
            print(f"{product.name} removed from the cart.")
    def get_total(self):
        total=0
        for product, qty in self.__items.items():
            total +=product.calculate_total(qty)
        return total
    def checkout(self):
        if not self.__items:
            print("Cart is empty. Nothing to checkout")
            return
        print("---------CART SUMMARY---------")
        total=self.get_total()
        for product, qty in self.__items.items():
            cost=product.calculate_total(qty)
            print(f"{product.name} x {qty} = Rs.{cost}")
        print("-"*40)
        print(f"    Total: Rs. {total}")
        print("-"*40)
        ch=input("Proceed to checkout? (y/n)")
        if ch.lower()=='y':
            for product, qty in self.__items.items():
                product.stock_update(qty)
            print(f"Order placed successfully! Amount paid: Rs. {total}")
            self.__items.clear()
            print("Cart cleared")            

In [26]:
# Create Products
tshirt = Clothing("C001", "Cotton T-Shirt", 500, 50, "L", "Cotton")
jeans = Clothing("C002", "Denim Jeans", 1500, 30, "XL", "Denim")
jacket = Clothing("C003", "Winter Jacket", 2000, 20, "M", "Wool")
book1 = Book("B001", "Python Programming", 450, 100, "John Doe", "978-1234567890", 350)
book2 = Book("B002", "Data Structures", 550, 80, "Jane Smith", "978-0987654321", 420)
# Display information
tshirt.display_info()
book1.display_info()
# calculate total price
print("CLOTHING:")
print(f"T-Shirt(L, 3 units) : Rs.{tshirt.calculate_total(3)}")
print(f"Jeans(XL, 2 units) : Rs.{jeans.calculate_total(2)}")
print(f"Jacket(M, 1 unit) : Rs.{jacket.calculate_total(1)}")
print("BOOKS:")
print(f"Python Book (3 units) : Rs.{book1.calculate_total(3)}")
print(f"Python Book (5 units) : Rs.{book1.calculate_total(5)}")
print(f"Python Book (10 units): Rs.{book1.calculate_total(10)}")
# ShoppingCart
cart = ShoppingCart()
# Add items
cart.add_item(tshirt, 3)
cart.add_item(book1, 6)
cart.add_item(jeans, 2)
cart.add_item(jacket, 25)
# Remove an item
cart.remove_item(jeans)
# Checkout
cart.checkout()
# Verify stock updated
print(tshirt)
print(book1)
# Try checkout on empty cart
cart.checkout()

CLOTHING:
T-Shirt(L, 3 units) : Rs.1500
Jeans(XL, 2 units) : Rs.3200
Jacket(M, 1 unit) : Rs.2000
BOOKS:
Python Book (3 units) : Rs.1350
Python Book (5 units) : Rs.2025.0
Python Book (10 units): Rs.4050.0
Sorry, insufficient stock for Winter Jacket
Denim Jeans removed from the cart.
---------CART SUMMARY---------
Cotton T-Shirt x 3 = Rs.1500
Python Programming x 6 = Rs.2430.0
----------------------------------------
    Total: Rs. 3930.0
----------------------------------------


Proceed to checkout? (y/n) y


Order placed successfully! Amount paid: Rs. 3930.0
Cart cleared

                Cotton T-Shirt (Rs. 500) - Stock: 47

                Python Programming (Rs. 450) - Stock: 94
Cart is empty. Nothing to checkout
